In [ ]:
%%writefile setup.sh

pip uninstall azureml-core -y
pip install azureml-core==0.1.0.* --index-url https://azuremlsdktestpypi.azureedge.net/sdk-release/master/588E708E0DF342C4A80BD954289657CF  --extra-index-url https://pypi.python.org/simple


pip uninstall azureml-contrib-datadrift -y
pip install azureml-datadrift==0.1.0.* --index-url https://azuremlsdktestpypi.azureedge.net/sdk-release/master/588E708E0DF342C4A80BD954289657CF  --extra-index-url https://pypi.python.org/simple

pip install azureml-opendatasets

In [1]:
import azureml.core

azureml.core.__version__

Failure while loading azureml_run_type_providers. Failed to load entrypoint hyperdrive = azureml.train.hyperdrive:HyperDriveRun._from_run_dto with exception (azureml-telemetry 1.0.57 (/home/cody/miniconda3/envs/azureml/lib/python3.7/site-packages), Requirement.parse('azureml-telemetry==0.1.0.5523758.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.PipelineRun = azureml.pipeline.core.run:PipelineRun._from_dto with exception (azureml-core 1.0.62 (/home/cody/miniconda3/envs/azureml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.5523758.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.ReusedStepRun = azureml.pipeline.core.run:StepRun._from_reused_dto with exception (azureml-core 1.0.62 (/home/cody/miniconda3/envs/azureml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.5523758.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.St

'1.0.62'

In [2]:
import os
from datetime import datetime, timedelta

import pandas as pd
from azureml.core import Dataset, Workspace, Run
from azureml.core.experiment import Experiment
from azureml.datadrift import DataDriftDetector
from azureml.opendatasets import NoaaIsdWeather

In [3]:
# Please type in your initials/alias. The prefix is prepended to the names of resources created by this notebook. 
prefix = "dd"

# optionally, set email address to receive an email alert for DataDrift
email_address = "cody.peterson@microsoft.com"

In [4]:
ws = Workspace.from_config()
print(ws.name, ws.resource_group, ws.location, ws.subscription_id, sep = '\n')

copeters930bugbash
copetersRG
eastus2euap
60582a10-b9fd-49f1-a546-c4194134bba8


In [5]:
# Generate baseline dataset
start_date = datetime(year=2019, month=4, day=1)
end_date = datetime(year=2019, month=4, day=15)

isd = NoaaIsdWeather(start_date,end_date)
baseline_dfdf = isd.to_pandas_dataframe().sample(100)
baseline_dfdf.to_csv("baseline_dataset.csv", index=False)

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2019/month=4/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2019/month=4/part-00102-tid-6212843797752751219-0a0ad45e-f97a-4e86-9019-4513057ceb40-462292.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=33336.13 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=33360.59 [ms]


In [6]:
# Upload baseline dataset to remote and create Dataset object 
dstore = ws.get_default_datastore()

dstore.upload_files(['./baseline_dataset.csv'], target_path='data/', overwrite=True, show_progress=True)

baseline_datastore_path = [(dstore, 'data/baseline_dataset.csv')]
baseline_dataset = Dataset.Tabular.from_delimited_files(path=baseline_datastore_path)
baseline_dataset

Uploading an estimated of 1 files
Uploading ./baseline_dataset.csv
Uploaded ./baseline_dataset.csv, 1 files out of an estimated total of 1
Uploaded 1 files


TabularDataset
{
  "source": [
    "('workspaceblobstore', 'data/baseline_dataset.csv')"
  ],
  "definition": [
    "GetDatastoreFiles",
    "ParseDelimited",
    "DropColumns",
    "SetColumnTypes"
  ]
}

In [7]:
# Generate target dataset
start_date = datetime(year=2019, month=7, day=1)
end_date = datetime(year=2019, month=7, day=15)

isd = NoaaIsdWeather(start_date,end_date)
target_df = isd.to_pandas_dataframe()
target_df = target_df.sort_values('datetime', ascending=True).groupby('day').head(1000)

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2019/month=7/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2019/month=7/part-00107-tid-6212843797752751219-0a0ad45e-f97a-4e86-9019-4513057ceb40-462298.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=36046.38 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=36058.46 [ms]


In [8]:
# Translate the original July timestamps to whatever the current date is minus 7 days (for backfill)
unique_timestamps = target_df.datetime.unique()
start_timestamp = pd.Timestamp(unique_timestamps[0])
curr_timestamp = pd.Timestamp.utcnow() - timedelta(days=7)
new_timestamps = []
for t in unique_timestamps:
    t_timestamp = pd.Timestamp(t)
    delta = t_timestamp.day - start_timestamp.day
    new_timestamp = curr_timestamp + timedelta(days=delta)
    new_timestamps.append(t_timestamp.replace(year=new_timestamp.year, month=new_timestamp.month, day=new_timestamp.day))
new_timestamps

for index, t in enumerate(unique_timestamps):
    target_df = target_df.replace(pd.Timestamp(t), new_timestamps[index])
target_df.head()

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,pastWeatherIndicator,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version
0,412164,99999,2019-09-23,25.027,55.366,50.0,90.0,2.6,29.0,NaN,...,NaN,NaN,NaN,NaN,DUBAI MINHAD AB,AE,412164-99999,2019,1,1.0
7314869,387500,99999,2019-09-23,37.467,53.967,-22.0,320.0,4.0,27.2,1007.1,...,NaN,NaN,NaN,NaN,ESENGULY,TX,387500-99999,2019,1,1.0
7314632,298070,99999,2019-09-23,53.350,75.450,94.0,NaN,NaN,12.0,1019.4,...,NaN,NaN,NaN,NaN,ERTIS,KZ,298070-99999,2019,1,1.0
124547,998193,99999,2019-09-23,21.433,-157.783,3.0,100.0,5.1,29.0,1014.1,...,NaN,NaN,NaN,NaN,MOKUOLOE,US,998193-99999,2019,1,1.0
7314384,576870,99999,2019-09-23,28.117,112.783,120.0,340.0,3.0,26.5,1005.0,...,NaN,NaN,NaN,NaN,CHANGSHA,CH,576870-99999,2019,1,1.0


In [9]:
# Create target dataset CSVs
if not os.path.exists('target_data/'):
    os.mkdir('target_data/')
        
files = []
days = target_df.day.unique().tolist()
for day in days:
    csv_df = target_df[target_df['day'] == day]
    csv_df_timestamp = pd.Timestamp(csv_df.datetime.unique()[0])
    
    year = csv_df_timestamp.year
    month = str(csv_df_timestamp.month).zfill(2)
    day = str(csv_df_timestamp.day).zfill(2)
    
    year_dir = 'target_data/{}'.format(year)
    if not os.path.exists(year_dir):
        os.mkdir(year_dir)
        
    month_dir = 'target_data/{}/{}'.format(year, month)
    if not os.path.exists(month_dir):
        os.mkdir(month_dir)
    file_path = "target_data/{}/{}/{}.csv".format(year, month, day)
    csv_df.to_csv(file_path, index=False, mode='w+')
    files.append(file_path)

In [10]:
# Upload target dataset to remote and create Dataset object

dstore.upload_files(files,
                    relative_root="target_data",
                    target_path='data2/target_dataset/', 
                    overwrite=True, 
                    show_progress=True)
target_datastore_path = [(dstore, 'data2/target_dataset/' + f.split('target_data/')[1]) for f in files]
target_dataset = Dataset.Tabular.from_delimited_files(path=target_datastore_path)
target_dataset = target_dataset.with_timestamp_columns(fine_grain_timestamp="datetime")
target_dataset

Uploading an estimated of 15 files
Uploading target_data/2019/09/23.csv
Uploading target_data/2019/09/24.csv
Uploading target_data/2019/09/25.csv
Uploading target_data/2019/09/26.csv
Uploading target_data/2019/09/27.csv
Uploading target_data/2019/09/28.csv
Uploading target_data/2019/09/29.csv
Uploading target_data/2019/09/30.csv
Uploading target_data/2019/10/01.csv
Uploading target_data/2019/10/02.csv
Uploading target_data/2019/10/03.csv
Uploading target_data/2019/10/04.csv
Uploading target_data/2019/10/05.csv
Uploading target_data/2019/10/06.csv
Uploading target_data/2019/10/07.csv
Uploaded target_data/2019/10/01.csv, 1 files out of an estimated total of 15
Uploaded target_data/2019/09/27.csv, 2 files out of an estimated total of 15
Uploaded target_data/2019/09/23.csv, 3 files out of an estimated total of 15
Uploaded target_data/2019/10/02.csv, 4 files out of an estimated total of 15
Uploaded target_data/2019/09/26.csv, 5 files out of an estimated total of 15
Uploaded target_data/2019

TabularDataset
{
  "source": [
    "('workspaceblobstore', 'data2/target_dataset/2019/09/23.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/09/24.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/09/25.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/09/26.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/09/27.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/09/28.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/09/29.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/09/30.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/10/01.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/10/02.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/10/03.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/10/04.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/10/05.csv')",
    "('workspaceblobstore', 'data2/target_dataset/2019/10/06.csv')",
   

# Test create_from_datasets and list

## Method Signatures
`create_from_datasets(workspace, name, baseline_dataset, target_dataset, compute_target_name=None, frequency=None, feature_list=None, alert_config=None, drift_threshold=None, latency=None)`

`list(workspace, model_name=None, model_version=None, baseline_dataset=None, target_dataset=None)`

## Additional Tests
- Check additional optional params for `create_from_datasets()`
- Mixed DatasetBased and ModelBased params should throw exception for `list()`

In [12]:
test_name = "sample_dd"
datadrift = DataDriftDetector.get_by_name(ws,test_name)

assert datadrift.workspace == ws
assert datadrift.baseline_dataset.id == baseline_dataset.id
assert datadrift.target_dataset.id == target_dataset.id

retrieved_dd_list = DataDriftDetector.list(ws, baseline_dataset=datadrift.baseline_dataset, target_dataset=datadrift.target_dataset)
retrieved_dd_list_2 = DataDriftDetector.list(ws, baseline_dataset=datadrift.baseline_dataset)
retrieved_dd_list_3 = DataDriftDetector.list(ws, target_dataset=datadrift.target_dataset)

assert len(retrieved_dd_list) == 1
assert len(retrieved_dd_list_2) == 1
assert len(retrieved_dd_list_3) == 1

retrieved_dd = retrieved_dd_list[0]
retrieved_dd_2 = retrieved_dd_list_2[0]
retrieved_dd_3 = retrieved_dd_list_3[0]

assert datadrift.name == retrieved_dd.name == retrieved_dd_2.name == retrieved_dd_3.name
assert datadrift._id == retrieved_dd._id == retrieved_dd_2._id == retrieved_dd_3._id
assert datadrift.baseline_dataset.id == retrieved_dd.baseline_dataset.id == retrieved_dd_2.baseline_dataset.id == retrieved_dd_3.baseline_dataset.id
assert datadrift.target_dataset.id == retrieved_dd.target_dataset.id == retrieved_dd_2.target_dataset.id == retrieved_dd_3.target_dataset.id

AssertionError: 

# Test get_by_name

## Method Signature
`get_by_name(workspace, name)`

## Additional Tests
- KeyError raised when DataDriftDetector doesn't exist

In [13]:
# Test get_by_name
retrieved_dd = DataDriftDetector.get_by_name(ws, test_name)
assert datadrift.name == retrieved_dd.name
assert datadrift._id == retrieved_dd._id
assert datadrift.baseline_dataset.id == retrieved_dd.baseline_dataset.id
assert datadrift.target_dataset.id == retrieved_dd.target_dataset.id

# Test enable_schedule and disable_schedule

## Method Signatures
`enable_schedule(self, create_compute_target=False)`
`disable_schedule(self)`

## Additional Tests
- N/A

In [14]:
from azureml.pipeline.core import Schedule

datadrift.enable_schedule(create_compute_target=True)
schedule_id = datadrift._schedule_id
schedule = Schedule.get(ws, schedule_id)
schedule._wait_for_provisioning(3600)

assert datadrift.enabled

datadrift.disable_schedule()

assert not datadrift.enabled

2019-09-30 14:17:03,401 - azureml.datadrift._logging._telemetry_logger.azureml.datadrift.datadriftdetector - ERROR - ActivityCompleted: Activity=enable_schedule, HowEnded=Failure, Duration=2899.62 [ms], Exception=TypeError
Traceback (most recent call last):
  File "/home/cody/miniconda3/envs/azureml/lib/python3.7/site-packages/azureml/_logging/chained_identity.py", line 38, in __init__
    super(ChainedIdentity, self).__init__(**kwargs)
TypeError: __init__() got an unexpected keyword argument '_create_in_cloud'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/cody/miniconda3/envs/azureml/lib/python3.7/site-packages/azureml/telemetry/activity.py", line 98, in log_activity
    yield activityLogger
  File "/home/cody/miniconda3/envs/azureml/lib/python3.7/site-packages/azureml/datadrift/datadriftdetector.py", line 886, in enable_schedule
    schedule = Schedule.get(self.workspace, self._schedule_id)
  File "/home/cody/mi

TypeError: __init__() got an unexpected keyword argument '_create_in_cloud'. Found key word arguments: dict_keys(['experiment', '_create_in_cloud']).

# Test run

## Method Signature
`run(self, target_date, services=None, compute_target_name=None, create_compute_target=False, feature_list=None, drift_threshold=None)`

## Additional Tests
- Ensure job passes

In [15]:
## NOTE: Set the current time to any date from 07/01/2019 to 07/14/2019
current_time = datetime.utcnow()
start = datetime(year=current_time.year, month=current_time.month, day=current_time.day)

test_name = "sample_run_dd"
datadrift = DataDriftDetector.create_from_datasets(ws, 
                                                   test_name, 
                                                   baseline_dataset=baseline_dataset, 
                                                   target_dataset=target_dataset)

run_id = datadrift.run(start, create_compute_target=True)
print("Run submitted with parent run id {}".format(run_id))
exp = Experiment(ws, datadrift._id)
run = Run(experiment=exp, run_id=run_id)
run.wait_for_completion(wait_post_processing=True)

metrics = datadrift._get_metrics()
print(metrics)

Run submitted with parent run id 2ea0c248-9d86-4833-8e41-b4c4144f1436_1569878241371


TypeError: 'NoneType' object is not iterable

# Test backfill

## Method Signature
`backfill(self, start_date, end_date, services=None, compute_target_name=None, create_compute_target=False,
          feature_list=None, drift_threshold=None)`

## Additional Tests
- ValueError thrown if start_date after end_date
- ValueError thrown if start_date and end_date range exceeds max backfill batch size of 30
- Ensure job passes

In [ ]:
test_name = "backfill_run_dd"

current_time = datetime.utcnow()
start = datetime(year=current_time.year, month=current_time.month, day=current_time.day)

backfill_start = start - timedelta(days=7)
backfill_end = backfill_start + timedelta(days=3)

datadrift = DataDriftDetector.create_from_datasets(ws,
                                                   test_name, 
                                                   baseline_dataset=baseline_dataset, 
                                                   target_dataset=target_dataset)

backfill_run = datadrift.backfill(backfill_start, backfill_end, create_compute_target=True)
backfill_run.wait_for_completion(wait_post_processing=True)